<a href="https://colab.research.google.com/github/ghizlane89/0__GenIA/blob/Bootcamp/W7_D2_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip install peft==0.4.0
%pip install datasets transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [2]:
# 🔄 Met à jour les bibliothèques concernées
%pip install -U datasets huggingface_hub fsspec


In [3]:
# -----------------------------------------------------------
# 📁 Création d’un dossier de cache pour stocker les résultats
# -----------------------------------------------------------
import os
os.makedirs("cache", exist_ok=True)

# -----------------------------------------------------------
# 🔍 Chargement du modèle pré-entraîné BLOOMZ-560M et du tokenizer associé
# -----------------------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# -----------------------------------------------------------
# 🗂️ Chargement du dataset et sélection d’un sous-échantillon pour le fine-tuning
# -----------------------------------------------------------
from datasets import load_dataset
full_data = load_dataset("Abirate/english_quotes", split="train")
data = full_data.select(range(int(0.005 * len(full_data))))  # 0,5 % du dataset

# -----------------------------------------------------------
# 🧹 Tokenisation des échantillons du dataset
# -----------------------------------------------------------
tokenized_data = data.map(lambda samples: tokenizer(samples["quote"], truncation=True, padding="max_length"), batched=True)

# -----------------------------------------------------------
# 🧪 Affichage de quelques échantillons tokenisés pour vérification
# -----------------------------------------------------------
train_sample = tokenized_data.select(range(5))
print(train_sample)

# -----------------------------------------------------------
# 🔧 Configuration de LoRA (Low-Rank Adaptation) pour adapter le modèle
# -----------------------------------------------------------
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],  # Cibles typiques pour BLOOMZ
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# -----------------------------------------------------------
# 🛠️ Application de LoRA sur le modèle de base
# -----------------------------------------------------------
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

# -----------------------------------------------------------
# ⚙️ Configuration des paramètres d’entraînement
# -----------------------------------------------------------
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

output_directory = os.path.join("cache", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,  # Taux d’apprentissage élevé pour convergence rapide
    num_train_epochs=3,
    use_cpu=True
)

# -----------------------------------------------------------
# 🎓 Initialisation du Trainer et lancement de l’entraînement
# -----------------------------------------------------------
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()

# -----------------------------------------------------------
# 💾 Sauvegarde du modèle fine-tuné avec horodatage
# -----------------------------------------------------------
import time
time_now = time.strftime("%Y-%m-%d-%H-%M-%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# -----------------------------------------------------------
# 🔮 Chargement du modèle fine-tuné pour l’inférence
# -----------------------------------------------------------
from peft import PeftModel
inference_model = PeftModel.from_pretrained(foundation_model, peft_model_path, is_trainable=False)

# -----------------------------------------------------------
# 🧠 Génération de texte à partir d’un prompt
# -----------------------------------------------------------
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = inference_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

# -----------------------------------------------------------
# 📝 Affichage du texte généré
# -----------------------------------------------------------
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

...

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.01757585078102687


Step,Training Loss


['Two things are infinite:  happiness or misbeing; when the world is your friend or bad friend.” “We can choose what we will not be afraid of the world, and it’s up to us, to get what we know. And we can control ourselves.” And']
